In [3]:
import cv2
import xml.etree.ElementTree as ET
import numpy as np

number = 1004

def create_mask_from_cvat_xml(xml_path, image_name, output_image_path, original_image_dimensions):
    """
    Creates a binary mask image from CVAT XML polygon annotations.
    0 = Land, 1 = Water
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()

    width, height = original_image_dimensions
    mask_image = np.zeros((height, width), dtype=np.uint8)  # single channel

    for image_tag in root.findall(f".//image[@name='{image_name}']"):
        for polygon_tag in image_tag.findall('polygon'):
            points_str = polygon_tag.get('points')
            label_name = polygon_tag.get('label')

            if not points_str:
                continue

            points = []
            for pair in points_str.split(';'):
                x, y = map(float, pair.split(','))
                points.append([int(x), int(y)])

            if label_name == 'Water':
                color = 1
            elif label_name == 'Land':
                color = 0
            else:
                continue

            cv2.fillPoly(mask_image, [np.array(points, np.int32)], color)

    cv2.imwrite(output_image_path, mask_image * 255)
    print(f"Mask image saved to: {output_image_path}")

# Example usage
create_mask_from_cvat_xml(f'annotations_{number}.xml', f'{number}.png', f'./manual_annotations/untouched/{number}.png', (512, 512))


Mask image saved to: ./manual_annotations/untouched/1004.png
